In [1]:
columns_classification = {
    "Restaurant Name": "Categorical (Nominal)",
    "Cuisine Type": "Categorical (Nominal)",
    "Average Cost for Two": "Numerical (Continuous)",
    "User Rating": "Numerical (Continuous)",
    "City": "Categorical (Nominal)",
    "Open/Closed Status": "Categorical (Nominal / Binary)"
}

for col, classification in columns_classification.items():
    print(f"- **{col}**: {classification}")

- **Restaurant Name**: Categorical (Nominal)
- **Cuisine Type**: Categorical (Nominal)
- **Average Cost for Two**: Numerical (Continuous)
- **User Rating**: Numerical (Continuous)
- **City**: Categorical (Nominal)
- **Open/Closed Status**: Categorical (Nominal / Binary)


In [2]:
import pandas as pd

# Define the dataset
data = {
    "App": [
        "Flipkart", "Flipkart", "IRCTC", "Instagram", 
        "Flipkart", "IRCTC", "Flipkart", "Instagram"
    ],
    "Data Field Example": [
        "Payment Method",
        "Delivery Speed Option",
        "Train Departure Time",
        "Number of Reel Views",
        "Product Rating (1-5)",
        "PNR Number",
        "Cart Total Amount (₹)",
        "Local Weather Temperature"
    ],
    "Measurement Scale": [
        "Nominal",
        "Ordinal",
        "Interval",
        "Ratio",
        "Ordinal",
        "Nominal",
        "Ratio",
        "Interval"
    ],
    "Explanation": [
        "Distinct categories with no mathematical order.",
        "Logical sequence from slowest to fastest, but intervals are non-uniform.",
        "Equal, meaningful differences, but 00:00 is an arbitrary reference point.",
        "True absolute zero (0 means no views) allowing multiplicative comparisons.",
        "Ranked customer feedback sequentially, but gaps between ratings are unequal.",
        "Unique identification label for tracking where digits hold no quantity.",
        "True zero (₹0) with valid mathematical operations and ratios.",
        "0°C is not an absolute absence of heat, but intervals between degrees are equal."
    ]
}

# Create a Pandas DataFrame
df = pd.DataFrame(data)

# Display the DataFrame
print(df.to_string(index=False))

      App        Data Field Example Measurement Scale                                                                      Explanation
 Flipkart            Payment Method           Nominal                                  Distinct categories with no mathematical order.
 Flipkart     Delivery Speed Option           Ordinal         Logical sequence from slowest to fastest, but intervals are non-uniform.
    IRCTC      Train Departure Time          Interval        Equal, meaningful differences, but 00:00 is an arbitrary reference point.
Instagram      Number of Reel Views             Ratio       True absolute zero (0 means no views) allowing multiplicative comparisons.
 Flipkart      Product Rating (1-5)           Ordinal     Ranked customer feedback sequentially, but gaps between ratings are unequal.
    IRCTC                PNR Number           Nominal          Unique identification label for tracking where digits hold no quantity.
 Flipkart     Cart Total Amount (₹)             Ratio  

In [3]:
import pandas as pd

# Sample Spotify listening history data
data = {
    "Date": ["2026-06-01", "2026-06-01", "2026-06-02", "2026-06-02", "2026-06-03"],
    "Song Name": ["Blinding Lights", "Anti-Hero", "As It Was", "Blinding Lights", "Levitating"],
    "Play Count per Day": [5, 3, 4, 6, 2]
}

# Create DataFrame
df = pd.DataFrame(data)

# 1. Convert 'Date' to datetime format for proper Time-Series handling
df["Date"] = pd.to_datetime(df["Date"])

# Display structure and data types
print("--- DataFrame Data Types ---")
print(df.dtypes)
print("\n--- Dataset Preview ---")
print(df)

# Example: Time-series & numerical aggregation (Total plays per day)
daily_plays = df.groupby("Date")["Play Count per Day"].sum()
print("\n--- Total Plays Per Day (Time-Series Aggregation) ---")
print(daily_plays)

# Example: Categorical analysis (Total plays per song)
song_plays = df.groupby("Song Name")["Play Count per Day"].sum()
print("\n--- Total Plays Per Song (Categorical Aggregation) ---")
print(song_plays)

--- DataFrame Data Types ---
Date                  datetime64[us]
Song Name                        str
Play Count per Day             int64
dtype: object

--- Dataset Preview ---
        Date        Song Name  Play Count per Day
0 2026-06-01  Blinding Lights                   5
1 2026-06-01        Anti-Hero                   3
2 2026-06-02        As It Was                   4
3 2026-06-02  Blinding Lights                   6
4 2026-06-03       Levitating                   2

--- Total Plays Per Day (Time-Series Aggregation) ---
Date
2026-06-01     8
2026-06-02    10
2026-06-03     2
Name: Play Count per Day, dtype: int64

--- Total Plays Per Song (Categorical Aggregation) ---
Song Name
Anti-Hero           3
As It Was           4
Blinding Lights    11
Levitating          2
Name: Play Count per Day, dtype: int64


In [5]:
def detect_measurement_scale(column_data):
  # Filter out nulls to get unique values and sample type
  valid_data = [x for x in column_data if x is not None]
  if not valid_data:
    return "Unknown"

  unique_vals = set(valid_data)
  n_unique = len(unique_vals)
  sample = valid_data[0]

  # Rule 1: Text-based (String) columns
  if isinstance(sample, str):
    # Check for common ordinal keywords (e.g., ratings, sizes, tiers)
    ordinal_keywords = {
        "low",
        "medium",
        "high",
        "small",
        "large",
        "poor",
        "good",
        "excellent",
        "standard",
        "express",
    }
    if any(str(v).lower() in ordinal_keywords for v in unique_vals):
      return "Ordinal"
    return "Nominal"

  # Rule 2: Numeric columns
  elif isinstance(sample, (int, float)):
    # Check if it looks like a bounded rating scale (e.g., 1-5 stars)
    if (
        all(isinstance(v, int) for v in valid_data)
        and n_unique <= 10
        and min(unique_vals) >= 0
        and max(unique_vals) <= 5
    ):
      return "Ordinal"

    # Check for negative numbers (indicates an arbitrary zero, like temperature or score offsets)
    if any(v < 0 for v in valid_data):
      return "Interval"

    # Default for positive numerical quantities/prices (true zero exists)
    return "Ratio"

  return "Unknown"